# 04 — Mock Sagemaker Training + Multi-Model Endpoint Demo

Companion notebook to `05-aws-sagemaker-deployment-and-autoscaling.md`.

This notebook does **not** call AWS at all — no credentials, no boto3 network calls. Instead it
implements small Python classes that mimic the *shape* of the real Sagemaker workflow
(`estimator.fit()`, `estimator.deploy()`) plus a mock multi-model endpoint router that dispatches
requests to the right in-memory "model," so you can build the mental model of how the pieces fit
together without needing an AWS account.

Fully offline, runs in well under a minute.


## 1. `FakeSagemakerEstimator` — mimics `sagemaker.huggingface.HuggingFace`

Real Sagemaker: `estimator.fit({"train": "s3://..."})` submits a managed training job on ephemeral
compute and blocks until it finishes; `estimator.deploy(...)` provisions a hosting endpoint and
returns a predictor. Here, `.fit()` actually trains a tiny real scikit-learn model on synthetic data
(so the demo is doing *something* real, not just printing), and `.deploy()` returns a mock endpoint
object -- no real infrastructure involved.


In [1]:
import time
import uuid
from dataclasses import dataclass, field


@dataclass
class FakeTrainingJobResult:
    job_name: str
    artifact_uri: str
    metrics: dict


class FakeSagemakerEstimator:
    """Mimics the shape of sagemaker.huggingface.HuggingFace / sagemaker.estimator.Estimator.

    Real usage:
        estimator = HuggingFace(entry_point="train.py", instance_type="ml.g4dn.xlarge", ...)
        estimator.fit({"train": "s3://bucket/train-data/"})
        predictor = estimator.deploy(initial_instance_count=1, instance_type="ml.m5.large")
    """

    def __init__(self, model_name: str, instance_type: str = "ml.m5.large"):
        self.model_name = model_name
        self.instance_type = instance_type
        self.trained_model = None
        self.last_job_result = None

    def fit(self, train_data_uri: str, X_train=None, y_train=None) -> FakeTrainingJobResult:
        """Mimics submitting + waiting on a Sagemaker training job."""
        job_name = f"{self.model_name}-job-{uuid.uuid4().hex[:8]}"
        print(f"[FakeSagemakerEstimator] Submitting training job '{job_name}'")
        print(f"  instance_type = {self.instance_type}")
        print(f"  train_data    = {train_data_uri}")

        if X_train is not None and y_train is not None:
            from sklearn.linear_model import LogisticRegression
            model = LogisticRegression(max_iter=1000)
            model.fit(X_train, y_train)
            self.trained_model = model
            train_acc = model.score(X_train, y_train)
            metrics = {"train_accuracy": round(train_acc, 4)}
        else:
            metrics = {"train_accuracy": None}

        artifact_uri = f"s3://claims-bucket/model-artifacts/{self.model_name}/{job_name}/model.tar.gz"
        print(f"  training complete -> artifact written to {artifact_uri}")
        print(f"  metrics: {metrics}\n")

        self.last_job_result = FakeTrainingJobResult(job_name, artifact_uri, metrics)
        return self.last_job_result

    def deploy(self, endpoint):
        """Registers this estimator's trained model onto a (fake) multi-model endpoint."""
        if self.trained_model is None:
            raise RuntimeError("Call .fit() before .deploy()")
        endpoint.register_model(self.model_name, self.trained_model, self.last_job_result.artifact_uri)
        print(f"[FakeSagemakerEstimator] Deployed '{self.model_name}' to endpoint '{endpoint.name}'")
        return endpoint


## 2. `FakeMultiModelEndpoint` — mimics a Sagemaker multi-model endpoint router

Real multi-model endpoints load models into memory on demand and evict least-recently-used ones
under memory pressure. This mock implements exactly that policy (a simple LRU cache of loaded
models keyed by name) plus request routing by `target_model`.


In [2]:
from collections import OrderedDict


class FakeMultiModelEndpoint:
    """Mimics a Sagemaker multi-model endpoint: many models behind one endpoint, on shared
    compute, with LRU-style loading/eviction and per-request target_model routing.

    Real usage:
        predictor.predict(data=payload, target_model="claim-classifier/v17/model.tar.gz")
    """

    def __init__(self, name: str, max_loaded_models: int = 2):
        self.name = name
        self.max_loaded_models = max_loaded_models
        self._registry = {}                     # model_name -> (model_obj, artifact_uri)
        self._loaded = OrderedDict()             # model_name -> model_obj, LRU-ordered
        self.invocation_log = []

    def register_model(self, model_name: str, model_obj, artifact_uri: str):
        self._registry[model_name] = (model_obj, artifact_uri)

    def _load(self, model_name: str):
        if model_name in self._loaded:
            self._loaded.move_to_end(model_name)  # mark as recently used
            return self._loaded[model_name]

        if model_name not in self._registry:
            raise KeyError(f"No such model registered on this endpoint: {model_name}")

        model_obj, artifact_uri = self._registry[model_name]
        if len(self._loaded) >= self.max_loaded_models:
            evicted_name, _ = self._loaded.popitem(last=False)  # evict least-recently-used
            print(f"  [endpoint] memory pressure -> evicted '{evicted_name}' from cache")

        print(f"  [endpoint] cold-loading '{model_name}' from {artifact_uri}")
        self._loaded[model_name] = model_obj
        return model_obj

    def predict(self, data, target_model: str):
        was_cached = target_model in self._loaded
        model_obj = self._load(target_model)
        prediction = model_obj.predict(data)
        self.invocation_log.append({"target_model": target_model, "cache_hit": was_cached})
        return prediction


## 3. Mock auto-scaling policy

A minimal stand-in for a Sagemaker target-tracking auto-scaling policy: given a current load metric
(e.g. invocations-per-instance), decide the desired instance count to keep that metric near a
target value.


In [3]:
class FakeAutoScalingPolicy:
    """Mimics a target-tracking scaling policy on SageMakerVariantInvocationsPerInstance."""

    def __init__(self, target_invocations_per_instance: float, min_capacity: int = 1, max_capacity: int = 6):
        self.target = target_invocations_per_instance
        self.min_capacity = min_capacity
        self.max_capacity = max_capacity

    def desired_instance_count(self, current_invocations_per_minute: float) -> int:
        raw = current_invocations_per_minute / self.target
        desired = max(self.min_capacity, min(self.max_capacity, round(raw)))
        return desired


policy = FakeAutoScalingPolicy(target_invocations_per_instance=70.0, min_capacity=1, max_capacity=6)

for load in [20, 70, 140, 300, 600]:
    n = policy.desired_instance_count(load)
    print(f"load = {load:>4} invocations/min -> desired instances = {n}")


load =   20 invocations/min -> desired instances = 1
load =   70 invocations/min -> desired instances = 1
load =  140 invocations/min -> desired instances = 2
load =  300 invocations/min -> desired instances = 4
load =  600 invocations/min -> desired instances = 6


## 4. End-to-end demo: train + deploy three modules to one multi-model endpoint


In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(0)


def make_toy_dataset(class_a_texts, class_b_texts):
    texts = class_a_texts + class_b_texts
    labels = [0] * len(class_a_texts) + [1] * len(class_b_texts)
    vec = TfidfVectorizer()
    X = vec.fit_transform(texts)
    return X, labels, vec


# Toy training data per module (tiny, just to make .fit() do something real)
claim_X, claim_y, claim_vec = make_toy_dataset(
    ["reduces symptom severity significantly", "improves patient outcomes markedly"],
    ["common adverse reaction reported", "boxed warning for cardiovascular risk"],
)
isi_X, isi_y, isi_vec = make_toy_dataset(
    ["important safety information section present and complete"] * 3,
    ["important safety information section missing or truncated"] * 3,
)

endpoint = FakeMultiModelEndpoint(name="claims-pipeline-endpoint", max_loaded_models=2)

claim_estimator = FakeSagemakerEstimator(model_name="claim-classifier")
claim_estimator.fit("s3://claims-bucket/training-data/claim-classifier/v17/", claim_X, claim_y)
claim_estimator.deploy(endpoint)

isi_estimator = FakeSagemakerEstimator(model_name="isi-classifier")
isi_estimator.fit("s3://claims-bucket/training-data/isi-classifier/v8/", isi_X, isi_y)
isi_estimator.deploy(endpoint)

print(f"\nModels registered on endpoint '{endpoint.name}': {list(endpoint._registry.keys())}")


[FakeSagemakerEstimator] Submitting training job 'claim-classifier-job-ce2bd0a9'
  instance_type = ml.m5.large
  train_data    = s3://claims-bucket/training-data/claim-classifier/v17/
  training complete -> artifact written to s3://claims-bucket/model-artifacts/claim-classifier/claim-classifier-job-ce2bd0a9/model.tar.gz
  metrics: {'train_accuracy': 1.0}

[FakeSagemakerEstimator] Deployed 'claim-classifier' to endpoint 'claims-pipeline-endpoint'
[FakeSagemakerEstimator] Submitting training job 'isi-classifier-job-574d2916'
  instance_type = ml.m5.large
  train_data    = s3://claims-bucket/training-data/isi-classifier/v8/
  training complete -> artifact written to s3://claims-bucket/model-artifacts/isi-classifier/isi-classifier-job-574d2916/model.tar.gz
  metrics: {'train_accuracy': 1.0}

[FakeSagemakerEstimator] Deployed 'isi-classifier' to endpoint 'claims-pipeline-endpoint'

Models registered on endpoint 'claims-pipeline-endpoint': ['claim-classifier', 'isi-classifier']


## 5. Route a few requests through the multi-model endpoint

Watch the cache behavior: the first call to each model is a cold load; calling the same model again
is a cache hit; with `max_loaded_models=2` and only two models registered here, nothing gets evicted
in this small demo -- registering a third model and calling it would trigger an eviction.


In [5]:
claim_query = claim_vec.transform(["reduces symptom severity by a large margin"])
isi_query = isi_vec.transform(["important safety information section present"])

print("Request 1 -> claim-classifier (cold load expected)")
pred1 = endpoint.predict(claim_query, target_model="claim-classifier")
print(f"  prediction: {pred1}\n")

print("Request 2 -> isi-classifier (cold load expected)")
pred2 = endpoint.predict(isi_query, target_model="isi-classifier")
print(f"  prediction: {pred2}\n")

print("Request 3 -> claim-classifier again (cache hit expected)")
pred3 = endpoint.predict(claim_query, target_model="claim-classifier")
print(f"  prediction: {pred3}\n")

print("Invocation log:")
for entry in endpoint.invocation_log:
    print(f"  {entry}")


Request 1 -> claim-classifier (cold load expected)
  [endpoint] cold-loading 'claim-classifier' from s3://claims-bucket/model-artifacts/claim-classifier/claim-classifier-job-ce2bd0a9/model.tar.gz
  prediction: [0]

Request 2 -> isi-classifier (cold load expected)
  [endpoint] cold-loading 'isi-classifier' from s3://claims-bucket/model-artifacts/isi-classifier/isi-classifier-job-574d2916/model.tar.gz
  prediction: [0]

Request 3 -> claim-classifier again (cache hit expected)
  prediction: [0]

Invocation log:
  {'target_model': 'claim-classifier', 'cache_hit': False}
  {'target_model': 'isi-classifier', 'cache_hit': False}
  {'target_model': 'claim-classifier', 'cache_hit': True}


## Takeaways

- `FakeSagemakerEstimator.fit()` / `.deploy()` mirror the real `sagemaker.huggingface.HuggingFace`
  API shape closely enough to build intuition for it, while actually training a tiny real
  scikit-learn model so the demo isn't purely cosmetic.
- `FakeMultiModelEndpoint` demonstrates the core multi-model endpoint trade-off directly: the first
  request for a model pays a "cold load" cost, subsequent requests for the same model are cheap
  (cache hit), and under memory pressure the least-recently-used model gets evicted -- exactly the
  dynamic chapter 05 describes conceptually.
- `FakeAutoScalingPolicy` shows target-tracking scaling isn't complicated math: pick a target metric
  value, and back out the instance count needed to keep observed load near that target, clipped to
  `[min_capacity, max_capacity]`.
- None of this requires AWS credentials or network access -- that's deliberate. The point is the API
  *shape* and the operational trade-offs, which transfer directly to reading/writing real
  `boto3`/`sagemaker` SDK code once you have AWS access.
